# ASTR 457: Foundations of Data Science in Astronomy

**Fall 2026, University of Illinois Urbana-Champaign**

Prof. Gautham Narayan | TA: Abha Vishwakarma

Tue Oct 27, 2026 - Day 18: Probabilistic graphical models and hierarchical Bayes

<img src="images/uiuc_logo.png" alt="University of Illinois Urbana-Champaign wordmark" width="220" style="display:block;margin:0 auto;">

In [ ]:
# Presenter setup (a RISE "skip" cell: it runs, but is not a slide).
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.linalg import cho_factor, cho_solve
from astropy.cosmology import FlatLambdaCDM
from ipywidgets import interactive_output, FloatSlider, Checkbox, HBox, VBox, Layout
from IPython.display import display
plt.rcParams.update({'font.size': 13, 'figure.figsize': (9.4, 3.6)})   # 9.4 in = 940 px fits a 13-inch MacBook at 1280 px; 7.4 in beside widget controls

# matplotlib's math-text parser is a shared singleton and is not thread-safe. ipykernel 7 handles slider messages on
# another thread (ipykernel #1564), so a redraw can overlap the next cell and break any "$...$" label
# ("Expected end of text", ipympl #610). A lock around the parser fixes that on any kernel version (ipympl PR #621).
import threading
try:
    from matplotlib import _mathtext
    if not getattr(_mathtext.Parser.parse, '_locked', False):
        _mathtext_lock = threading.Lock()
        _unlocked_parse = _mathtext.Parser.parse
        def _locked_parse(self, *args, **kwargs):
            with _mathtext_lock:
                return _unlocked_parse(self, *args, **kwargs)
        _locked_parse._locked = True
        _mathtext.Parser.parse = _locked_parse
except (ImportError, AttributeError):
    print("could not lock matplotlib's math-text parser: do not move a slider while another cell is running")

def controls_right(fn, **widgets):
    # the plot on the left, its controls stacked on the right (the Day 12 layout)
    for w in widgets.values():
        w.layout = Layout(width='230px')
        w.style = {'description_width': '75px'}
    display(HBox([interactive_output(fn, widgets), VBox(list(widgets.values()), layout=Layout(justify_content='center', width='240px'))]))

# NASA GISTEMP v4 global annual mean anomaly (lecture/08/data/, the Day 15 and Day 17 file); 2026 is incomplete
_s = pd.read_csv('../08/data/gistemp_v4_GLB.Ts+dSST.csv', skiprows=1)
_s = _s[_s['J-D'] != '***']
yr, anom = _s['Year'].to_numpy(float), _s['J-D'].astype(float).to_numpy()

# Mrk 876 = PG 1613+658, a Palomar-Green quasar (ZTF18aaiywzv, g band, ALeRCE API copy of 2026-10-26; the CSV header has the provenance): in nobody's Lab 07 live set
_q = pd.read_csv('data/mrk876_g.csv', comment='#')
_q = _q[_q['sigma_mag'] < 0.1]
t_q, m_q, e_q = _q['mjd'].to_numpy(), _q['mag_g'].to_numpy(), _q['sigma_mag'].to_numpy()


def hessian(f, p, h=1e-4):
    n = len(p); H = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            ei = np.zeros(n); ej = np.zeros(n); ei[i] = h; ej[j] = h
            H[i, j] = (f(p + ei + ej) - f(p + ei - ej) - f(p - ei + ej) + f(p - ei - ej)) / (4 * h * h)
    return H

# Day 10's relation: Reines & Volonteri 2015, ApJ 813, 82, Table 1 - 244 broad-line AGN, log M* and log M_BH (lecture/05/data, week 5).
# Their adopted measurement errors: 0.3 dex on log M* (their Sec. 2.4) and 0.5 dex on log M_BH (Sec. 2.3); their Kelly 2007 fit,
# pivoted at log M* = 11: slope 1.05 +- 0.11, intercept 7.45 +- 0.08, intrinsic scatter 0.24 dex (their Sec. 4.1, Eq. 5).
_rv = pd.read_csv('../05/data/reines2015_table1.csv', comment='#')
x_rv, y_rv = _rv['log_Mstar'].to_numpy(float), _rv['log_MBH'].to_numpy(float)
SX_RV, SY_RV, X0_RV = 0.3, 0.5, 11.0
def nll_plain_rv(p):
    # log M* treated as exact: y ~ N(a (x - 11) + b, sigma_int^2 + 0.5^2)
    a, b, ls = p
    v = np.exp(2 * ls) + SY_RV**2
    r = y_rv - (a * (x_rv - X0_RV) + b)
    return 0.5 * np.sum(r**2 / v + np.log(2 * np.pi * v))
def nll_latent_rv(p):
    # log M* latent, drawn from N(mu_x, sigma_pop): the Sep 29 2x2 for every galaxy, (x_obs, y_obs) jointly Gaussian
    a, b, ls, mx, lsp = p
    si2, sp2 = np.exp(2 * ls), np.exp(2 * lsp)
    Cm = np.array([[sp2 + SX_RV**2, a * sp2], [a * sp2, a * a * sp2 + si2 + SY_RV**2]])
    d = np.stack([x_rv - mx, y_rv - (a * (mx - X0_RV) + b)], axis=1)
    return 0.5 * (np.einsum('ni,ij,nj->', d, np.linalg.inv(Cm), d) + x_rv.size * (np.log(np.linalg.det(Cm)) + 2 * np.log(2 * np.pi)))
_o = dict(xatol=1e-7, fatol=1e-7, maxiter=20000)
fit_plain_rv = minimize(nll_plain_rv, [0.5, 7.0, np.log(0.3)], method='Nelder-Mead', options=_o)
fit_latent_rv = minimize(nll_latent_rv, [1.0, 7.4, np.log(0.3), x_rv.mean(), np.log(0.35)], method='Nelder-Mead', options=_o)
err_plain_rv = np.sqrt(np.diag(np.linalg.inv(hessian(nll_plain_rv, fit_plain_rv.x))))
err_latent_rv = np.sqrt(np.diag(np.linalg.inv(hessian(nll_latent_rv, fit_latent_rv.x))))
a_p, a_l = fit_plain_rv.x[0], fit_latent_rv.x[0]
sp_rv, si_rv = np.exp(fit_latent_rv.x[4]), np.exp(fit_latent_rv.x[2])
print(f"Reines & Volonteri 2015: {x_rv.size} AGN; log M* exact: slope {a_p:.2f} +- {err_plain_rv[0]:.2f}, sigma_int {np.exp(fit_plain_rv.x[2]):.2f}; "
      f"log M* latent: slope {a_l:.2f} +- {err_latent_rv[0]:.2f}, sigma_int {si_rv:.2f}, mu_x {fit_latent_rv.x[3]:.2f}, sigma_pop {sp_rv:.2f}; "
      f"expected attenuation sigma_pop^2/(sigma_pop^2 + 0.3^2) = {sp_rv**2 / (sp_rv**2 + SX_RV**2):.2f}; their Kelly 2007 fit 1.05 +- 0.11")

# The population for the shrinkage demo: one value with its error per object (the CSV header has the provenance)
# GWTC-3 confident binary black holes (both masses > 3 Msun): source-frame chirp mass with its 90% interval from GWOSC (CC BY 4.0);
# the demo's Gaussian error bar is the mean half-width divided by 1.645
_p = pd.read_csv('data/gwtc3_bbh_masses.csv', comment='#')
_p = _p.sort_values('chirp_mass_source').reset_index(drop=True)                                              # sorted by chirp mass for the plot
pop_name, pop_y = _p['event'].astype(str).str.slice(0, 8).to_numpy(), _p['chirp_mass_source'].to_numpy(float)   # GW191103, the date part of the name
pop_e = 0.5 * (_p['chirp_mass_source_err_lo'].to_numpy(float) + _p['chirp_mass_source_err_hi'].to_numpy(float)) / 1.645
print(f"population demo: {len(pop_y)} objects, binary black hole chirp masses, values {pop_y.min():.1f} to {pop_y.max():.1f}, median error {np.median(pop_e):.2f}")

## Thursday - we blazed through GPs in the last twenty minutes, largely to help with Lab 07

How is Lab 07 going?

### Let's step back and go over GPs a bit more slowly.

<img src="images/day17_static_gp_fit.png" alt="Thursday's Gaussian-process fit to the GISTEMP temperature record: the data, the GP mean and its band, from the Day 17 widget" style="display:block;margin:0 auto;max-height:230px;max-width:100%">

* we don't have a parametric model for Lab 07's quasar, but that's OK, we can write down a parametric model for the covariance itself
* two points close in time are correlated and two far apart aren't - this function of the lag, $k(\Delta t)$, that describes the covariance between two epochs is called the **kernel** function

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**So we don't have a parametric model for the light curve, but we do have a parametric model for its covariance - what can we do with this thing?**

</div>

## The Gaussian process prior

<img src="images/day17_static_gp_prior.png" alt="Five random curves drawn from a Gaussian process prior (left) and the kernel they obey, a falling curve of agreement against time lag (right); the Day 17 widget" style="display:block;margin:0 auto;max-height:300px;max-width:100%">

* for a fixed kernel, but no data, you have a Gaussian process **prior** - an infinite number of curves can be drawn from it (five of them are on the left)
* the kernel has two parameters, an amplitude $\sigma$ (how big the wiggles are) and a timescale $\tau$ (how long one lasts)
* the curve on the right is $k(\Delta t)$ at Thursday's values

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**For a fixed kernel, but no data, you have a Gaussian process prior - a probability distribution over functions.**

</div>

In [ ]:
# Day 17's GISTEMP fit at its maximum-likelihood kernel, redrawn (a "skip" cell: runs, not a slide)
_y = anom - anom.mean()
def _kse(a, b, sigma, tau):
    return sigma ** 2 * np.exp(-0.5 * ((a[:, None] - b[None, :]) / tau) ** 2)
def _nll_gp(p, x, y):
    sigma, tau, s = np.exp(p)
    c = cho_factor(_kse(x, x, sigma, tau) + s ** 2 * np.eye(x.size))
    return 0.5 * y @ cho_solve(c, y) + np.log(np.diag(c[0])).sum() + 0.5 * x.size * np.log(2 * np.pi)
_gp = minimize(_nll_gp, np.log([0.5, 20, 0.1]), args=(yr, _y), method='Nelder-Mead')
sig_ml, tau_ml, s_ml = np.exp(_gp.x)


In [ ]:
# Five prior guesses over the GISTEMP points, then five posterior curves through them (a "skip" cell)
def plot_prior_vs_posterior():
    rng = np.random.default_rng(18)
    xs = np.linspace(1880, 2035, 300)
    K_pp = _kse(xs, xs, sig_ml, tau_ml) + 1e-8 * np.eye(xs.size)
    prior = rng.multivariate_normal(np.zeros(xs.size), K_pp, 5)
    c = cho_factor(_kse(yr, yr, sig_ml, tau_ml) + s_ml ** 2 * np.eye(yr.size))
    Ks = _kse(xs, yr, sig_ml, tau_ml)
    m = Ks @ cho_solve(c, _y)
    cov = K_pp - Ks @ cho_solve(c, Ks.T)
    post = rng.multivariate_normal(m, (cov + cov.T) / 2, 5)
    sd = np.sqrt(np.clip(np.diag(cov), 0, None))
    fig, axs = plt.subplots(1, 2, figsize=(9.4, 3.2), sharey=True)
    for ax, curves, title in zip(axs, (prior, post), ('the prior: five draws, no data yet', f'the posterior: sigma {sig_ml:.2f} K, tau {tau_ml:.0f} yr, noise {s_ml:.2f} K')):
        ax.plot(yr, anom, 'k.', ms=3, zorder=3, label='GISTEMP annual mean')
        for cv in curves:
            ax.plot(xs, cv + anom.mean(), lw=1, alpha=0.8)
        ax.set_xlabel('year'); ax.set_title(title, fontsize=11); ax.set_ylim(-1.6, 2.2)
    axs[1].plot(xs, m + anom.mean(), 'C0', lw=2, label='GP mean'); axs[1].fill_between(xs, m - sd + anom.mean(), m + sd + anom.mean(), color='C0', alpha=0.25, label='confidence band')
    axs[1].legend(fontsize=8, loc='upper left')
    axs[0].set_ylabel('global anomaly [K]')
    plt.tight_layout(); plt.show()

In [ ]:
# Two epochs under Lab 07's kernel: the joint Gaussian, and what measuring one does to the other (a "skip" cell)
def plot_conditional(lag=100.0, tau=865.0, sig=0.31, y1_obs=0.35):
    rho = np.exp(-lag / tau)
    fig, (ax, axr) = plt.subplots(1, 2, figsize=(9.4, 3.3), gridspec_kw={'width_ratios': [1.1, 1]})
    g = np.linspace(-1, 1, 300); Y1, Y2 = np.meshgrid(g, g)
    Q = (Y1**2 - 2 * rho * Y1 * Y2 + Y2**2) / (sig**2 * (1 - rho**2))
    ax.contourf(Y1, Y2, np.exp(-0.5 * Q), levels=12, cmap='Blues')
    ax.axvline(y1_obs, color='C3', lw=2, label=f'measured $y_1$ = {y1_obs:.2f}')
    ax.set_xlabel('$y_1$, brightness at epoch 1 [mag from the mean]'); ax.set_ylabel('$y_2$, brightness {:.0f} d later'.format(lag)); ax.set_aspect('equal')
    ax.set_title(f'the joint Gaussian, correlation {rho:.2f}', fontsize=11); ax.legend(fontsize=9, loc='upper left')
    m2, s2 = rho * y1_obs, sig * np.sqrt(1 - rho**2)
    axr.plot(g, np.exp(-0.5 * (g / sig)**2) / (sig * np.sqrt(2 * np.pi)), 'k', lw=1.5, label=f'$y_2$ before: 0 $\\pm$ {sig:.2f}')
    axr.plot(g, np.exp(-0.5 * ((g - m2) / s2)**2) / (s2 * np.sqrt(2 * np.pi)), 'C3', lw=2, label=f'$y_2$ given $y_1$: {m2:.2f} $\\pm$ {s2:.2f}')
    axr.set_xlabel('$y_2$ [mag from the mean]'); axr.set_ylabel('probability density'); axr.legend(fontsize=9); axr.set_title('the slice along the red line', fontsize=11)
    plt.tight_layout(); plt.show()

## Two epochs at a time: what measuring one does to the other

In [ ]:
# two epochs 100 days apart under Lab 07's kernel (amplitude 0.31 mag, timescale 865 d, the numbers we'll fit to a quasar in ten minutes)
plot_conditional()

* two epochs close in time are correlated, so measuring $y_1$ teaches us about $y_2$ - the probability density for $y_2$ collapses to the conditional distribution, the slice along the red line
* the slice is narrower than what we started with and its centre has moved toward the measurement, and that is all "conditioning on the data" ever does
* with $N$ epochs it is the same picture in $N$ dimensions: slice at every measured point, and the posterior over the curve is what comes out

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Measuring one epoch collapses what we believe about the next to the conditional distribution.**

</div>

## Constraining the GP

In [ ]:
# the same kernel before and after the 146 GISTEMP points (NASA's global temperature record, Thursday's data):
# most of the prior draws miss the data, the posterior draws all pass through it
plot_prior_vs_posterior()

* left is five draws from the prior, and on the right see how the posterior curves all pass through the data, i.e. we've conditioned the model on the data (evaluated the posterior)
* those curves are the posterior, and their average is the blue line and their spread is the band
* the three numbers in the title are the amplitude $\sigma$ and the timescale $\tau$, which are in the prior, and the noise, which is in the likelihood
* $\sigma$ and $\tau$ are parameters of a prior distribution - we call these **hyperparameters**

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**You "condition" the model on the data (i.e. keep only the curves that pass through it) and what comes out is the posterior.**

</div>

## We don't actually observe the true brightness

* the light curve is the true brightness plus noise, $y = f(t) + \epsilon$, and we never observe $f$, which is **latent** (unobserved)
* as you've seen, there isn't one single $f$ but infinitely many for a given kernel, and we **marginalize** over them (integrate over every one rather than pick one) - $f$ is behaving like a parameter
* the likelihood of the data under the GP model is simply one $N$-dimensional Gaussian, $\mathcal{N}(\mathbf{y} \mid \mu, C)$, with the error bars added on the diagonal of $C$
* this is not really any different from the Lab 03 likelihood (the only new thing is that $C$ has off-diagonal elements)

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**i.e. the observed light curve is a noisy realisation of a latent (unobserved) true brightness - we marginalize over it, and the likelihood is one big Gaussian.**

</div>

## Today's plan

**What does a straight-line fit subtly assume, and where else does the same fix apply?**

* finish the GP recap: how you draw from the covariance matrix, a quasar to fiddle with, the PGM for a GP
* probabilistic graphical models, a very useful way of visualizing a generative model
* going hierarchical: the quasar, a population of black holes, and today's colloquium
* last 12 minutes: **Saumya Patel** (Part 2) and **Austin Mathison** (Parts 1 and 3) defend the midterm
* Lab 07 is due tomorrow (Wed Oct 28) by Noon, and Lab 08 will be posted on Thursday, starting from the graph of your midterm

In [ ]:
# How a draw is made from a covariance matrix: six epochs, Lab 07's kernel, Cholesky (a "skip" cell)
def plot_cholesky():
    t6 = np.array([0, 20, 50, 400, 420, 800.0])                      # six epochs: two close pairs and a loner
    C6 = 0.3**2 * np.exp(-np.abs(t6[:, None] - t6[None, :]) / 800.0)    # C_ij = k(t_i - t_j): the DRW kernel at 0.3 mag, 800 d, no noise term
    L6 = np.linalg.cholesky(C6)                                       # C = L L^T, L lower-triangular; the same factor gives ln|C| = 2 sum(ln L_ii) and C^-1 r by two triangular solves, so Lab 07's ln L never inverts C
    z6 = np.random.default_rng(23).normal(size=6)                     # six independent standard normals
    y6 = L6 @ z6                                                      # one draw: correlated the way C says
    fig, axs = plt.subplots(1, 3, figsize=(9.4, 3.0), gridspec_kw={'width_ratios': [1, 1, 1.3]})
    for ax, M, name in zip(axs[:2], (C6, L6), ('C  (mag$^2$)', 'L, with $LL^T = C$')):
        ax.imshow(M, cmap='viridis', vmin=0, vmax=C6.max())
        for i in range(6):
            for j in range(6):
                if M[i, j] != 0:
                    ax.text(j, i, f'{M[i, j]:.3f}'[1:] if abs(M[i, j]) < 1 else f'{M[i, j]:.2f}', ha='center', va='center', fontsize=8, color='w' if M[i, j] < 0.5 * C6.max() else 'k')
        ax.set_xticks(range(6)); ax.set_yticks(range(6)); ax.set_xticklabels([f'{int(v)}' for v in t6], fontsize=9); ax.set_yticklabels([f'{int(v)}' for v in t6], fontsize=9)
        ax.set_title(name, fontsize=11); ax.set_xlabel('day', fontsize=9)
    ax = axs[2]
    ax.stem(t6, z6 * 0.3, linefmt='gray', markerfmt='o', basefmt=' ', label='0.3 z: independent')
    ax.plot(t6, y6, 'C3s-', ms=6, label='L z: one draw')
    ax.axhline(0, color='k', lw=0.5); ax.set_xlabel('day'); ax.set_ylabel('mag from the mean'); ax.legend(fontsize=8, loc='upper right')
    ax.set_title('y = L z', fontsize=11)
    plt.tight_layout(); plt.show()

## So if you have a covariance matrix, how do you draw from it?

$$\mathbf{y} = \mu + L\,\mathbf{z}, \qquad L L^{\mathsf T} = C, \qquad \mathbf{z} \sim \mathcal{N}(0, 1)^N$$

* $C_{ij} = k(t_i - t_j)$ is the kernel evaluated at the lag between epochs $i$ and $j$
* $L$ is the lower-triangular matrix square root of $C$ (the Cholesky factor, `np.linalg.cholesky`)
* $\mathbf{z}$ is $N$ independent standard normals, and $\textrm{Cov}(L\mathbf{z}) = L L^{\mathsf T} = C$

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**$L$ is the square root of the covariance matrix, and $L\mathbf{z}$ is one draw from it.**

</div>

## Six epochs under Lab 07's kernel

In [ ]:
# amplitude 0.3 mag, timescale 800 d: the matrix, its square root, and one draw built from six independent numbers
plot_cholesky()

* the close epochs (days 0, 20, 50 and 400, 420) are strongly correlated and day 800 is only weakly correlated with anything, and the off-diagonal elements are there to say which epochs are correlated
* Lab 07 adds the error bars on the diagonal and reuses this matrix $L$ when it computes $\ln L$, once per choice of $\sigma$ and $\tau$

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**The close epochs come out together because the off-diagonal elements of $C$ say they must.**

</div>

In [ ]:
# Worked example: Mrk 876 under a GP you control (a "skip" cell)
tq0 = t_q - t_q.min()
def plot_quasar():
    fig, ax = plt.subplots(figsize=(8.0, 2.4))
    ax.errorbar(tq0, m_q, e_q, fmt='.', color='k', ms=3, elinewidth=0.6)
    ax.invert_yaxis(); ax.set_xlabel('days since first epoch'); ax.set_ylabel('g [mag]')
    ax.set_title(f'Mrk 876 (PG 1613+658): {len(t_q)} ZTF g-band epochs over {(t_q.max() - t_q.min()) / 365.25:.1f} yr', fontsize=12)
    plt.tight_layout(); plt.show()

KERNELS = {
    'damped random walk (Lab 07)': lambda d, tau: np.exp(-np.abs(d) / tau),
    'Matern 3/2':                  lambda d, tau: (1 + np.sqrt(3) * np.abs(d) / tau) * np.exp(-np.sqrt(3) * np.abs(d) / tau),
    'squared exponential':         lambda d, tau: np.exp(-0.5 * (d / tau)**2),
    'periodic, P = 365 d':         lambda d, tau: np.exp(-2 * np.sin(np.pi * d / 365.25)**2 / (tau / 365.25)**2),
}
def kmat(a, b, kind, sig, tau):
    return sig**2 * KERNELS[kind](a[:, None] - b[None, :], tau)

def nll_gp(p, kind, t, y, e):
    # the Lab 07 Part 2 log-likelihood for any of the four rules: mean mu, amplitude sigma, timescale tau, quoted errors on the diagonal
    mu, ls, lt = p
    sig, tau = np.exp(ls), np.exp(lt)
    if not (0.01 < sig < 3 and 10 <= tau <= 4000):
        return 1e30
    c = cho_factor(kmat(t, t, kind, sig, tau) + np.diag(e**2))
    r = y - mu
    return 0.5 * r @ cho_solve(c, r) + np.log(np.diag(c[0])).sum() + 0.5 * t.size * np.log(2 * np.pi)
_starts = [[m_q.mean(), np.log(0.1), np.log(50.0)], [m_q.mean(), np.log(0.5), np.log(1000.0)]]      # two starts, as the lab asks
_fits = [minimize(nll_gp, s0, args=('damped random walk (Lab 07)', tq0, m_q, e_q), method='Nelder-Mead', options=dict(xatol=1e-5, fatol=1e-5)) for s0 in _starts]
_best = min(_fits, key=lambda f: f.fun)
mu_q, sig_q, tau_q = _best.x[0], np.exp(_best.x[1]), np.exp(_best.x[2])
lnL_q = -_best.fun
print(f"DRW fit to Mrk 876: mu = {mu_q:.3f} mag, sigma = {sig_q:.3f} mag, tau = {tau_q:.0f} d, ln L = {lnL_q:.1f}; both starts agree to {abs(_fits[0].fun - _fits[1].fun):.2f} in ln L")

_order = np.argsort(tq0)
_rank = np.random.default_rng(457).permutation(tq0.size)        # a fixed random order, so "fraction" adds points the same way every time
_tgrid = np.linspace(-60, tq0.max() + 300, 700)

def plot_gp_widget(kind, sig, tau, fraction, draws):
    use = _rank < int(round(fraction * tq0.size))
    fig = plt.figure(figsize=(7.4, 4.3))
    gs = fig.add_gridspec(2, 2, width_ratios=[1, 1.6], height_ratios=[2.2, 1.1], hspace=0.08, wspace=0.5, left=0.07, right=0.99, top=0.93, bottom=0.13)
    axm = fig.add_subplot(gs[:, 0]); axl = fig.add_subplot(gs[0, 1]); axr = fig.add_subplot(gs[1, 1], sharex=axl)
    # left: the correlation between every pair of epochs under this rule, on a fixed 0-1 colour scale
    ts = tq0[_order]
    corr = KERNELS[kind](ts[:, None] - ts[None, :], tau)
    im = axm.imshow(corr, vmin=0, vmax=1, cmap='viridis', aspect='equal')
    axm.set_title('correlation between epochs', fontsize=10); axm.set_xlabel('epoch (time order)'); axm.set_ylabel('epoch (time order)')
    axm.tick_params(labelsize=8)
    fig.colorbar(im, ax=axm, fraction=0.046, pad=0.22, location='bottom').ax.tick_params(labelsize=8)
    # right: the light curve, the points used, and the posterior under these numbers (no fitting: you are the fit)
    axl.errorbar(tq0[~use], m_q[~use], e_q[~use], fmt='.', color='C3', ms=3, elinewidth=0.4, alpha=0.7, label='not used')
    if use.sum() > 0:
        axl.errorbar(tq0[use], m_q[use], e_q[use], fmt='.', color='k', ms=3, elinewidth=0.5, label=f'used: {use.sum()}')
        c = cho_factor(kmat(tq0[use], tq0[use], kind, sig, tau) + np.diag(e_q[use]**2))
        Ks = kmat(_tgrid, tq0[use], kind, sig, tau)
        m = mu_q + Ks @ cho_solve(c, m_q[use] - mu_q)
        cov = kmat(_tgrid, _tgrid, kind, sig, tau) - Ks @ cho_solve(c, Ks.T)
        sd = np.sqrt(np.clip(np.diag(cov), 0, None))
        Kd = kmat(tq0, tq0[use], kind, sig, tau)                      # the posterior mean at every epoch, used or not
        m_d = mu_q + Kd @ cho_solve(c, m_q[use] - mu_q)
        sd_d = np.sqrt(np.clip(sig**2 - np.einsum('ij,ji->i', Kd, cho_solve(c, Kd.T)), 0, None))
        lnL = -nll_gp([mu_q, np.log(sig), np.log(tau)], kind, tq0[use], m_q[use], e_q[use])
        lnL_ref = -nll_gp([mu_q, np.log(sig_q), np.log(tau_q)], 'damped random walk (Lab 07)', tq0[use], m_q[use], e_q[use])
    else:
        m = np.full(_tgrid.size, mu_q); cov = kmat(_tgrid, _tgrid, kind, sig, tau); sd = np.sqrt(np.diag(cov)); lnL = lnL_ref = None
        m_d = np.full(tq0.size, mu_q); sd_d = np.full(tq0.size, sig)
    axl.plot(_tgrid, m, 'C0', lw=1.2, label='posterior mean'); axl.fill_between(_tgrid, m - sd, m + sd, color='C0', alpha=0.25, label='confidence band')
    if draws:
        for d in np.random.default_rng(3).multivariate_normal(m, (cov + cov.T) / 2 + 1e-8 * np.eye(_tgrid.size), 3):
            axl.plot(_tgrid, d, lw=0.6, alpha=0.7)
    axl.set_ylim(m_q.max() + 0.25, m_q.min() - 0.25); axl.set_ylabel('g [mag]'); axl.tick_params(labelbottom=False)
    # the residuals: data minus the posterior mean, at every epoch; the band is the posterior spread there
    r = m_q - m_d
    axr.fill_between(tq0[_order], -sd_d[_order], sd_d[_order], color='C0', alpha=0.25)
    axr.errorbar(tq0[~use], r[~use], e_q[~use], fmt='.', color='C3', ms=3, elinewidth=0.4, alpha=0.7)
    if use.sum() > 0:
        axr.errorbar(tq0[use], r[use], e_q[use], fmt='.', color='k', ms=3, elinewidth=0.5)
    axr.axhline(0, color='C0', lw=1); axr.set_ylim(0.45, -0.45); axr.set_xlabel('days since first epoch'); axr.set_ylabel('resid [mag]', fontsize=9)
    rms = ', '.join(f'{lab} {np.sqrt(np.mean(r[sel]**2)):.3f}' for lab, sel in (('used', use), ('not used', ~use)) if sel.sum())
    axr.text(0.01, 0.93, f'rms data - mean [mag]: {rms}', transform=axr.transAxes, fontsize=8, va='top')
    axl.legend(fontsize=7, loc='lower left', ncol=2)
    axl.set_title(f'sigma {sig:.2f} mag, tau {tau:.0f} d: ln L = {lnL:.0f}  (fitted DRW: {lnL_ref:.0f})' if lnL is not None else 'no points used: this is the prior', fontsize=9)
    plt.show()

def gp_controls():
    from ipywidgets import Dropdown, FloatLogSlider
    controls_right(plot_gp_widget,
                   kind=Dropdown(options=list(KERNELS), value='damped random walk (Lab 07)', description='kernel'),
                   sig=FloatSlider(value=0.5, min=0.05, max=0.6, step=0.01, description='amplitude'),                  # opens at a first guess, not the fit: the cue slides it to the fit
                   tau=FloatLogSlider(value=100.0, base=10, min=1, max=3.5, step=0.05, description='timescale'),
                   fraction=FloatSlider(value=0.5, min=0, max=1, step=0.1, description='fraction'),
                   draws=Checkbox(value=False, description='3 draws'))

## A full worked example of a GP: Mrk 876

In [ ]:
# the light curve, with its seasons and gaps
plot_quasar()

* Mrk 876 (PG 1613+658), a Seyfert 1 at $z = 0.13$, with 1176 ZTF $g$-band points over 7.6 years
* from the same public ZTF archive (ALeRCE) as your Lab 07 objects, but not in anyone's set
* no known period and no parametric model for the light curve - but that's OK, we write down a parametric model for the covariance instead

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**A GP is not "model-free": the kernel is the model, it's just not a physically parametrized one.**

</div>

## So how do you choose the kernel $k$ and its hyperparameters? Lets fiddle:

* start at Lab 07's kernel with a first guess (0.50 mag, 100 d) and half the data
* slide to the best-fit values (0.31 mag, ~865 d) and see what happens to $\ln L$, the log-likelihood in the title
* then fraction 0, then 1, then timescale 50 d, then the squared exponential kernel

In [ ]:
# left: the correlation matrix the kernel gives the 1176 epochs, on a fixed 0-1 scale
# right: the posterior given the points used
gp_controls()

<img src="images/day18_static_gp_widget.png" alt="The GP widget on Mrk 876 at the Lab 07 kernel, the fitted amplitude and timescale, half the points used: left the correlation matrix between epochs on a fixed 0 to 1 colour scale, bright near the diagonal and in the blocks of each observing season; right the light curve with the used points black, the rest red, and the posterior mean and its band" style="max-height:330px; display:block; margin:0 auto;">

(static copy of the widget for readers on GitHub)

* notice that what we've written down is a model for how the observations are correlated with each other - the left panel is the correlation matrix, so the amplitude divides out of it and only shows in the width of the band
* with no points the band is just the prior, with all of them it's the best-fit DRW - note how it collapses onto the data in between!
* the bottom panel is the residuals, data minus the posterior mean, and if there is structure left in them (or red points outside the band) the kernel or the hyperparameters are wrong
* *training* the GP means maximising $\ln L$ (the title) with respect to the hyperparameters

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**We are writing down a model for how the observations are correlated with each other, and $\ln L$ is a measure of how well that model explains the data.**

</div>

## Probabilistic graphical model for a GP

<img src="images/day18_pgm_drw.png" alt="The damped-random-walk model as a graph: three fitted parameters mu, sigma and tau, the fixed epochs and errors, and one grey node holding all N magnitudes together with no plate" style="display:block;margin:0 auto;max-height:260px;max-width:100%">

<div style="font-size:0.8em; text-align:center; color:#666;">Blue circles are what we fit, $\mu$ the mean, $\sigma$ the amplitude, $\tau$ the timescale. Dots are fixed, the epochs and the quoted error bars $\sigma_1 \ldots \sigma_N$ (a different thing from the amplitude $\sigma$, same letter).</div>

* three circles we fit ($\mu$, $\sigma$ and $\tau$), two sets of fixed dots (the epochs and the error bars), and one shaded circle for the measured data, all $N$ magnitudes in one node
* with time-series the data are not conditionally independent (knowing one magnitude tells you about the next)
* i.e. you don't have a nice plate (the box that says 'repeat for every object')
* every epoch is correlated with every other, and the $N$ magnitudes are one draw from an $N$-dimensional Gaussian
* the true brightness $f$ isn't on the graph because we don't actually observe it, and marginalizing over it is what gave us $\mathcal{N}(\mathbf{y} \mid \mu, C)$

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**A PGM is a very useful way of visualizing a generative model, and every model you've fit this term has one.**

</div>

## Probabilistic graphical models

<img src="images/day18_pgm_grammar.png" alt="Key to a probabilistic graphical model: blue circle a fitted parameter, white circle a latent true value, gray circle an observed value, black dot a fixed quantity, arrows for conditional dependence, a plate for repetition over objects" style="display:block;margin:0 auto;max-height:330px;max-width:100%">

* in statistics these are called **probabilistic graphical models** (PGMs), and the Sep 29 review test had you draw one - here is the key
* circles are PDFs, an arrow is a conditional relationship (B given A), and a plate is a repeated model component whose contents are conditionally independent (one object tells you nothing about another)
* Day 12's lens graph (Padma's Fig. 1) had three levels

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Draw it the way you would generate mock data, from the population down to the noise.**

</div>

In [ ]:
# Day 10's relation and the two straight-line fits to it (a "skip" cell)
def plot_reines(latent=False):
    fig, ax = plt.subplots(figsize=(7.6, 3.1))
    ax.errorbar(x_rv, y_rv, xerr=SX_RV, yerr=SY_RV, fmt='.', color='k', ms=4, elinewidth=0.3, alpha=0.5, label=f'{x_rv.size} broad-line AGN (Reines & Volonteri 2015), their 0.3 and 0.5 dex errors')
    xs = np.linspace(8.7, 11.7, 50)
    ax.plot(xs, a_p * (xs - X0_RV) + fit_plain_rv.x[1], 'C3', lw=2, label=f'log M* taken as exact: slope {a_p:.2f}')
    if latent:
        ax.plot(xs, a_l * (xs - X0_RV) + fit_latent_rv.x[1], 'C0', lw=2, label=f'log M* latent: slope {a_l:.2f}')
        ax.plot(xs, 1.05 * (xs - 11) + 7.45, 'C0', ls=':', lw=1.2, label='their Kelly 2007 fit: slope 1.05')
    ax.set_xlabel('log M* [Msun]'); ax.set_ylabel('log M_BH [Msun]'); ax.set_xlim(8.7, 11.7); ax.set_ylim(4.8, 9.0)
    ax.legend(fontsize=8, loc='upper left')
    plt.tight_layout(); plt.show()

## Day 10's relation, black hole mass against stellar mass, as a graph

<img src="images/day18_pgm_reines_plain.png" alt="The straight-line fit as Day 10 did it, drawn as a graph: three fitted parameters, the slope, the intercept and the intrinsic scatter, above a plate of 244 galaxies; inside the plate the observed log black hole mass has arrows from the three parameters and from two fixed dots, the galaxy stellar mass and its measurement error" style="display:block;margin:0 auto;max-height:250px;max-width:100%">

* Reines & Volonteri 2015: the 244 broad-line AGN behind the figure Day 10 showed - here is a straight-line fit to them with $\log M_*$ taken as exact, the way Day 10 did OLS on its simulated sample
* 3 blue circles, 1 gray circle, 2 black dots, 1 plate, and that is the whole fit

**Which arrow is missing?** (hint: it was on the Sep 29 review test)

In [ ]:
# the real sample, with the straight-line fit that takes log M* as exact
plot_reines()

## Measurement errors in $\log M_*$

<img src="images/reines2015_mbh_mstar.png" alt="Reines and Volonteri 2015 Figure 8 left: black-hole mass against host stellar mass for broad-line AGN, dwarf AGN and dynamical black holes, with four published relations drawn as lines" style="display:block;margin:0 auto;max-height:260px;max-width:100%">

<div style="font-size:0.8em; text-align:center; color:#666;">Reines &amp; Volonteri 2015, ApJ 813, 82, Fig. 8 (left), the figure Day 10 showed.</div>

* $\log M_*$ is a black dot, i.e. fixed - but Reines &amp; Volonteri adopt 0.3 dex errors on it (and 0.5 dex on $\log M_{\rm BH}$)
* errors in $x$ bias the slope toward zero (Day 10's attenuation), and making the error bar on $y$ bigger doesn't remove that bias
* the missing arrow is true $\log M_* \to$ observed $\log M_*$, with 0.3 dex on it

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Write down your assumptions!!!** (the arrows you didn't draw are assumptions too)

</div>

## The relation with the missing arrow

<img src="images/day18_pgm_sep29_redrawn.png" alt="The Sep 29 review-test reference graph in today's colors: a population mean and width above a latent true x per object, which feeds the observed x and the latent true y; alpha, beta and the intrinsic scatter feed the true y" style="display:block;margin:0 auto;max-height:300px;max-width:100%">

<div style="font-size:0.8em; text-align:center; color:#666;">The Sep 29 review test's graph, redrawn in today's colors: here $x$ is $\log M_*$, $y$ is $\log M_{\rm BH}$, and $\sigma_{x,i}$ is the 0.3 dex.</div>

* each galaxy has a true $\log M_*$ and a true $\log M_{\rm BH}$ that we don't observe, $2 \times 244$ **latent** (unobserved) variables, the white circles
* each true $\log M_*$ is drawn from $\mathcal{N}(\mu_x, \sigma_{\rm pop})$, which is a prior on it with hyperparameters $\mu_x$ and $\sigma_{\rm pop}$ that we fit
* the observed $\log M_*$ and $\log M_{\rm BH}$ are the true values plus noise (hence the two arrows down to the gray circles)

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**If it has an error bar, it has a true value we don't observe (a latent variable, and a white circle in the graph).**

</div>

## Reading a graph as a likelihood

$$p(\textrm{everything}) = \prod_{\textrm{circles}} p(\textrm{circle} \mid \textrm{the arrows into it})$$

* a gray circle is a likelihood term (the measured value given the parameters feeding into it), and a white circle with arrows in is the prior on that object's latent variable
* the straight line with $\log M_*$ exact ($\alpha$ the slope, $\beta$ the value at $\log M_* = 11$, $\sigma_{\rm int}$ the scatter about the line): $\prod_i \mathcal{N}\!\left(y_i \mid \alpha\,(x_i - 11) + \beta,\ \sigma_{\rm int}^2 + \sigma_{y}^2\right) \times p(\alpha)\,p(\beta)\,p(\sigma_{\rm int})$
* and you can go the other way, since every term in your $\ln L$ is an arrow

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Build up complexity by layering conditional probabilities.**

</div>

## Marginalizing out the latent variables

$$p(\textrm{obs}_i) = \int p(\textrm{obs}_i \mid \textrm{true}_i)\, p(\textrm{true}_i \mid \mu_x, \sigma_{\rm pop})\, d\,\textrm{true}_i \qquad \textrm{Sep 29, Q3:}\ \begin{pmatrix} \sigma_{\rm pop}^2 + \sigma_{x,i}^2 & \alpha\sigma_{\rm pop}^2 \\ \alpha\sigma_{\rm pop}^2 & \alpha^2\sigma_{\rm pop}^2 + \sigma_{\rm int}^2 + \sigma_{y,i}^2 \end{pmatrix}$$

* every factor is Gaussian, and so is the integral
* the $2\times2$ is the covariance of one galaxy's observed $(x, y)$ pair - the Sep 29 review test had you type it ($\alpha$ is this slope), and the population width adds to the error bars in quadrature
* $2 \times 244$ latent values marginalized out analytically, leaving $\alpha$, $\beta$, $\sigma_{\rm int}$ and the two population hyperparameters $\mu_x$, $\sigma_{\rm pop}$
* your midterm's Part 2 is the same thing with three numbers per supernova, a $3\times3$ instead of this $2\times2$ - Lab 08 has you draw that graph

* here $\sigma_{\rm pop}$ comes out at 0.33 dex against the 0.3 dex error, so how much do you expect the slope to be biased?

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**And if you really don't care about the parameters for individual measurements, just marginalize over them.**

</div>

## The same 244 AGN with latent $\log M_*$

In [ ]:
# the exact-x fit next to the latent-x fit, on the same data
plot_reines(latent=True)

* the slope goes from 0.55 with $\log M_*$ exact to 0.92 with it latent, and its error bar grows (0.07 to 0.14), because we no longer treat $\log M_*$ as exact
* the predicted ratio of the two slopes is $\sigma_{\rm pop}^2 / (\sigma_{\rm pop}^2 + \sigma_x^2)$ = 0.55 (that it equals the red slope is a coincidence), and the measured ratio is 0.60
* Reines &amp; Volonteri's own fit with Kelly 2007 (a fit with errors in both variables) gives 1.05 $\pm$ 0.11, about 1 sigma from ours

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Note how the slope goes up by two thirds once the errors in $\log M_*$ are in the model!**

</div>

## Back to the quasar's graph

<img src="images/day18_pgm_drw.png" alt="The damped-random-walk model as a graph again: three fitted parameters, fixed epochs and errors, one grey node holding all N magnitudes, no plate" style="display:block;margin:0 auto;max-height:230px;max-width:100%">

* the true brightness isn't drawn (we marginalized over it), but it would go between the hyperparameters and the data, like the white circles on the Sep 29 graph
* the relation, once marginalized, is the same idea but keeps its plate (one gray node per galaxy with the $2\times2$ into it) - the quasar is one node because its epochs are not independent

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**This is a multi-level model.**

</div>

## Going hierarchical

If you have multiple sets of measurements, each of which are related, we may want to learn something not just about each individual measurement, *but also the population*.

* fit a model to each one individually, assuming they share no similarities
* or pool all the data and estimate one model, assuming they are all identical
* or **hierarchical modeling** (multi-level, the quasar's word), which gives the best of both worlds by modeling what they share but also letting you estimate each one's own parameters
* in symbols: $\theta$ is one object's own parameters, $\phi$ the population's (the hyperparameters)

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**$p(\theta \mid \phi)$ contains interesting physics, and getting values for $\phi$ given the data (the individuals) can help us understand the population.**

</div>

## What a gravitational-wave detector measures: the chirp

<img src="images/day18_gw150914_fig1.png" alt="GW150914, the first detection: the strain at LIGO Hanford and Livingston over 0.2 seconds, the reconstructed waveform, the residuals, and a time-frequency map in which the signal sweeps up from 35 to 250 Hz as the two black holes spiral in" style="display:block;margin:0 auto;max-height:300px;max-width:100%">

<div style="font-size:0.8em; text-align:center; color:#666;">Abbott et al. 2016, PRL 116, 061102, Fig. 1 (CC BY 3.0): GW150914, the first detection, at Hanford and Livingston.</div>

* two black holes spiral together and merge, and the last fraction of a second is what LIGO's two detectors record as a strain of $10^{-21}$, a change in the 4 km arms far smaller than a proton
* the frequency sweeps up as they close in (the chirp), and how fast it sweeps is set by one combination of the two masses, the **chirp mass** $\mathcal{M} = (m_1 m_2)^{3/5} / (m_1 + m_2)^{1/5}$ - so that is the number the waveform measures best, and the two masses separately are poorly known
* GWTC-3 is the LIGO-Virgo-KAGRA catalog for the second half of the third observing run (Abbott et al. 2023): 35 confident events, and we keep the 32 with both masses above $3\,M_\odot$, i.e. two black holes

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Each merger gives us one well-measured number, its chirp mass, and 32 of them are a population.**

</div>

## A population: binary black hole chirp masses

<img src="images/day18_pgm_pooling.png" alt="Three graphs for 32 objects: complete pooling with one mass mu for all, no pooling with an independent theta_j per object, and partial pooling where each theta_j is drawn from a population with mean mu and width sigma_pop" style="display:block;margin:0 auto;max-height:280px;max-width:100%">

* 32 binary black holes from GWTC-3 (the catalog before Day 16's GWTC-4), each with one number $y_j$ and its error $\sigma_j$
* $y_j$ is each event's chirp mass from the slide before, and $\sigma_j$ its error
* pool all the data and you get one number (**complete pooling**), or fit each event individually, assuming they share no similarities, and get 32 numbers (**no pooling**)
* the third graph (**partial pooling**) keeps $\mu$, adds $\sigma_{\rm pop}$ and turns every $\theta_j$ white (2 hyperparameters above 32 latent masses)

<div style="font-size:0.8em; text-align:center; color:#666;">Chirp mass = 0.87 of either mass when the two are equal. Error bars: the 90% interval from GWOSC, the Gravitational Wave Open Science Center, symmetrized and divided by 1.645, i.e. 1 sigma (data CC BY 4.0, gwosc.org). The spread is sigma_pop, not Lab 07's tau. My group fits hierarchical models to GWTC-3 too, to test general relativity (Xie, Narayan and Yunes 2025, arXiv:2510.02515).</div>

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Which graph you draw is a modeling choice, and each one assumes something different about what the 32 black holes share.**

</div>

In [ ]:
# Partial pooling with everything Gaussian: the posterior of each object's true value given sigma_pop (a "skip" cell)
# With a flat prior on the population mean this is all closed form (Gelman et al. 2013, BDA3, Sec. 5.5, Eqs. 5.17 and 5.20-5.21)
def _given_spread(tau, y, s):
    w = 1 / (s**2 + tau**2)
    mu = np.sum(w * y) / np.sum(w); V_mu = 1 / np.sum(w)
    B = s**2 / (s**2 + tau**2)                        # the weight that pulls each object toward mu
    theta = y - B * (y - mu)
    sd = np.sqrt((1 - B) * s**2 + B**2 * V_mu)
    return mu, np.sqrt(V_mu), theta, sd, B
def _spread_posterior(y, s, taus):
    lp = []
    for t in taus:                                    # marginal posterior of the spread, uniform prior
        w = 1 / (s**2 + t**2); m = np.sum(w * y) / np.sum(w)
        lp.append(0.5 * np.log(1 / np.sum(w)) + np.sum(-0.5 * np.log(s**2 + t**2) - 0.5 * w * (y - m)**2))
    p = np.exp(np.array(lp) - max(lp)); p /= np.trapezoid(p, taus)
    return p
_span = pop_y.max() - pop_y.min()
_taus = np.linspace(0.01 * _span, _span, 600)
_p_tau = _spread_posterior(pop_y, pop_e, _taus)
tau_median = _taus[np.searchsorted(np.cumsum(_p_tau) * (_taus[1] - _taus[0]), 0.5)]
tau_max = float(np.round(_span, 1))

def plot_population_pooling(tau, show_spread_posterior):
    mu, smu, th, sd, B = _given_spread(tau, pop_y, pop_e)
    n = len(pop_y)
    fig, (ax, ax2) = plt.subplots(1, 2, figsize=(7.4, 3.8), gridspec_kw={'width_ratios': [2.2, 1]})
    j = np.arange(n)
    ax.errorbar(pop_y, j + 0.18, xerr=pop_e, fmt='o', color='C3', ms=4, capsize=2, label='each object alone')
    ax.errorbar(th, j - 0.18, xerr=sd, fmt='s', color='C0', ms=4, capsize=2, label=f'partial pooling, spread = {tau:.1f}')
    for k in j:
        ax.plot([pop_y[k], th[k]], [k + 0.18, k - 0.18], color='gray', lw=0.6)
    ax.axvline(mu, color='C0', ls='--', lw=1); ax.axvspan(mu - smu, mu + smu, color='C0', alpha=0.1)
    ax.set_yticks([]); ax.invert_yaxis(); ax.set_xlabel('chirp mass [solar masses]'); ax.set_ylabel('32 events, sorted')
    for k in (np.argmax(pop_e), np.argmax(pop_y)):               # name the two that move most: the noisiest and the heaviest
        ax.annotate(pop_name[k], (pop_y[k] + pop_e[k], k + 0.18), fontsize=8, ha='left', va='center', xytext=(4, 0), textcoords='offset points')
    ax.legend(fontsize=9, loc='upper center', bbox_to_anchor=(0.5, -0.2), ncol=2, frameon=False)
    ax.set_title(f'population mean {mu:.1f} +- {smu:.1f}', fontsize=12)
    if show_spread_posterior:
        ax2.plot(_taus, _p_tau, 'k'); ax2.axvline(tau, color='C0', lw=1)
        ax2.set_xlabel(r'$\sigma_{\rm pop}$'); ax2.set_yticks([]); ax2.set_title(r'$p(\sigma_{\rm pop} \mid y)$, median %.1f' % tau_median, fontsize=11)
    else:
        ax2.axis('off'); ax2.text(0.5, 0.5, 'tick the box', ha='center', va='center', transform=ax2.transAxes, color='gray')
    plt.tight_layout(); plt.show()

def pooling_controls():
    controls_right(plot_population_pooling, tau=FloatSlider(value=3.0, min=0, max=tau_max, step=tau_max / 80, description='spread'),
                   show_spread_posterior=Checkbox(value=False, description='spread posterior'))

## Partial pooling: the spread $\sigma_{\rm pop}$

* let's slide the spread $\sigma_{\rm pop}$: it opens at 3, where every noisy event moves - then 0, then all the way right, then tick the box and slide to where the data put it, about 11

In [ ]:
# every object's true value under the population prior, and where the data put the spread
pooling_controls()

<img src="images/day18_static_pooling.png" alt="The population demo at the posterior median of the spread: each object's raw value with its error bar next to its partially pooled posterior, pulled toward the population mean; right panel the marginal posterior of the spread" style="max-height:330px; display:block; margin:0 auto;">

(static copy of the widget for readers on GitHub)

* at spread = 0 the events are all identical at the $1/\sigma_j^2$-weighted mean, 8.4 (the light events have the tiny error bars and win), and as spread $\to \infty$ every event stays where it started
* in between, the noisy and the extreme events move toward the population mean, and this is **shrinkage**
* tick the box and the data fit $\sigma_{\rm pop}$ too (about 11): most of the 32 barely change, and GW200220 goes from 62 to about 40 only because we assumed one Gaussian (more on Thursday)

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**The population helps make inference on the individual.**

</div>

## The GWTC-3 population paper

<img src="images/day18_gwtc3_masses.png" alt="GWTC-3 population paper Fig. 10: the inferred merger rate per unit primary mass, with peaks near 10 and 35 solar masses, and per unit mass ratio" style="display:block;margin:0 auto;max-height:300px;max-width:100%">

<div style="font-size:0.8em; text-align:center; color:#666;">Abbott et al. 2023, PRX 13, 011048, Fig. 10 (CC BY 4.0). Left: the merger rate (mergers per year per volume) per unit primary mass $m_1$; right: per unit mass ratio $q = m_2/m_1$.</div>

* Day 16's stellar graveyard put every merger as a dot on a mass axis, with nothing on the other axis
* the population paper fits what the masses are drawn from, and finds a power law plus a peak near 35 M$_\odot$
* they fit the heavier mass $m_1$ and we fit the chirp mass, so our single Gaussian with 2 hyperparameters is the simplest version of their model
* Thursday, Oct 29: a $p(\theta \mid \phi)$ you cannot integrate by hand (`PyMC`), and what the detectors miss (selection effects)

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Once we discover an object, we look for more to characterize their properties and understand their origin.**

</div>

## Colloquium today, 3:45 pm, this room: Jiayi Sun (Kentucky)

<img src="images/day18_sun_portrait_ngc4303.jpg" alt="Jiayi Sun, and NGC 4303 imaged with MUSE on the VLT and ALMA: ionized gas and stars in color with the cold molecular gas overlaid" style="display:block;margin:0 auto;max-height:300px;max-width:100%">

* **"Molecular Cloud Scale 'Micro-physics' as Drivers of the Baryon Cycle and Galaxy Evolution"**
* PHANGS (Physics at High Angular resolution in Nearby GalaxieS) maps the cold gas in nearby galaxies, 100 pc at a time
* refreshments at 3:30 pm in Astronomy 222 (go!)

<div style="font-size:0.8em; text-align:center; color:#666;">Photo: University of Kentucky Physics &amp; Astronomy. NGC 4303: ESO/ALMA (ESO/NAOJ/NRAO)/PHANGS, CC BY 4.0. Listing: astro.illinois.edu/news-events/astrophysics-colloquium.</div>

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Clouds inside galaxies inside a survey would fit the same partial-pooling graph with a second plate.**

</div>

## Wrapping up: what did the straight-line fit subtly assume?

* it left out the arrow from the true $\log M_*$ to the observed one (it treated a noisy variable as exact), and the slope came out 0.55 instead of 0.92
* the fix each time is a true value per object (a white circle) drawn from a population: the quasar's true brightness, 32 black holes, clouds in Jiayi Sun's galaxies
* hierarchical modeling is a statistically rigorous way to make scientific inferences about a population (or a specific object) based on many individuals (or observations)
* Thursday, Oct 29: hierarchical models in `PyMC`, selection effects, and Lab 08 starts from the graph of your midterm

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**Draw the graph, and the assumptions you didn't write down show up as missing arrows.**

</div>

## Before you go

* the midterm solution notebook and its example data set are posted; we're not going to walk through it in class, so you should review it
* Lab 07 is due **tomorrow, Wed Oct 28, by Noon**; every date is in `ASSIGNMENTS.md`
* Lab 08 posts Thursday, Oct 29: hierarchical Bayes, starting from the graph of your midterm's Part 2, due Wed Nov 4 (tentative until it posts)
* Thursday's defenses: **Chayil Ewing** and **Austin Huwer**, on Lab 07
* Jiayi Sun at 3:45 today, this room
* reading: Gelman et al., *Bayesian Data Analysis* 3rd ed., Ch. 5 (free at stat.columbia.edu/~gelman/book); Hilbe, de Souza and Ishida, *Bayesian Models for Astrophysical Data*, Ch. 8

## Midterm defenses, now: Saumya Patel, Austin Mathison

* the midterm in two halves: **Saumya** defends Part 2 (the supernovae), **Austin** defends Parts 1 and 3 (error bars, the referee report)
* 4 questions each, one per slot (walk-through, a choice, a check, a stretch), about 90 seconds a slot
* everyone else stays and listens, since this is the format you get when it's your lab

## EXTRA: why does this work?

Not presented today (the classic shrinkage example).

<img src="images/day18_loredo_efron_morris_diagram.png" alt="Efron and Morris 1975: batting averages of 18 players after 45 at-bats (red), their full-season averages (blue), and the shrinkage estimates (cyan); the shrunk estimates are closer to the truth, RMS error 0.148 against 0.277" style="display:block;margin:0 auto;max-height:260px;max-width:100%">

<div style="font-size:0.8em; text-align:center; color:#666;">Courtesy Tom Loredo (the slide I showed in ASTR 596 in 2020); the data are Efron and Morris 1975, JASA 70, 311; the RMS errors are the slide's.</div>

* 18 players with 45 at-bats each, and shrunk toward the mean the early-season averages end up almost 2x closer to the full-season ones
* all 18 players are humans playing baseball - they are members of a population, not arbitrary, unrelated binomial random number generators!
* but information that is relevant in the absence of data for player $i$ remains relevant when we additionally obtain that data - shrinkage estimators account for this

## EXTRA: fitting the same GP with `george`

Not presented today.

```python
import george
from george import kernels
gp = george.GP(0.3**2 * kernels.ExpKernel(metric=800.0**2), mean=y.mean(), fit_mean=True)   # the damped random walk; george's metric is tau^2
gp.compute(t, yerr)                              # builds and factorizes C
nll = lambda p: -(gp.set_parameter_vector(p) or gp.log_likelihood(y))
best = minimize(nll, gp.get_parameter_vector())  # mu, ln sigma^2, ln tau^2
```

* `george` on this light curve: $\mu$ = 15.07, $\sigma$ = 0.31 mag, $\tau$ = 861 d (cf. 865 d from our numpy version, the likelihood is flat along $\tau$ there)
* it fits $\ln \sigma^2$ and $\ln \tau^2$, i.e. a flat prior in george's parameters is the lab's log-uniform prior (and `celerite2` does the same thing in $O(N)$ instead of $O(N^3)$)
* not in the course environment (`pip install george`), and Lab 07 has you do it in numpy for transparency, so you see every term

## EXTRA: going hierarchical, the formula

Not presented today.

$$p(\theta \mid D) \propto p(D \mid \theta)\,p(\theta) \quad\longrightarrow\quad p(\phi, \theta \mid D) \propto p(D \mid \theta)\,p(\theta \mid \phi)\,p(\phi)$$

* $\theta$ = the parameters of one object, $\phi$ = the hyperparameters of the population, and $p(\theta \mid \phi)$ is the prior on $\theta$ (its parameters are the things we fit)
* for the relation, $\theta$ was each galaxy's true $\log M_*$ and $\phi$ was $(\mu_x, \sigma_{\rm pop})$; for your midterm's Part 2 it is each supernova's true $(x_1, c)$ and four hyperparameters
* Kelly 2007 (a regression with errors in both variables) is this graph with a mixture of Gaussians for $p(\theta \mid \phi)$

## EXTRA: the graph as `PyMC` code, and the supernova literature

Not presented today (the `PyMC` version is Thursday's demo).

```python
with pm.Model():
    mu_x, sig_x = pm.Normal('mu_x', 0, 2), pm.HalfNormal('sig_x', 2)          # the population
    x1_true = pm.Normal('x1_true', mu_x, sig_x, shape=N)                      # one latent stretch per supernova
    pm.Normal('x1_obs', x1_true, sn['x1_err'], observed=sn['x1'])              # the arrow into the gray circle
```

* one line per circle, and `shape=N` is the plate
* the cosmology fits all give each supernova a latent stretch and color (March et al. 2011, MNRAS 418, 2308; Rubin et al. 2015, ApJ 813, 137; Shariff et al. 2016, ApJ 827, 1)
* BayeSN (Mandel et al. 2022, MNRAS 510, 3939) adds a level above the light curve

<img src="images/day18_agrawal2025_bayesn.png" alt="Agrawal et al. 2025 Fig. 4: the distance modulus of the lensed supernova H0pe under each of seven cluster lens models, with SALT3-NIR and BayeSN light-curve fits side by side against the Pantheon+ and DES 5-year bands" style="display:block;margin:0 auto;max-height:260px;max-width:100%">

<div style="font-size:0.8em; text-align:center; color:#666;">Agrawal et al. 2025, arXiv:2510.07637 (UIUC), Fig. 4: SN H0pe's distance modulus per lens model from SALT3-NIR and from BayeSN; the authors find the lens-model magnifications biased high against the Pantheon+ and DES bands. Fair use with citation.</div>